# Check the OCI notebook environment
Run this small check before the longer gallery benchmarks. It verifies that a Jupyter client can submit work and that workers can import the notebook libraries.

In [ ]:
import os
from scaler import Client
scheduler_address = os.environ.get("SCALER_SCHEDULER_ADDRESS", "tcp://127.0.0.1:8516")
object_storage_address = os.environ.get("SCALER_OBJECT_STORAGE_ADDRESS", "tcp://127.0.0.1:8517")


In [ ]:
def check_worker(task):
    import importlib
    import socket
    import time
    from importlib.metadata import version
    for module in ("numpy", "scipy", "pandas", "parfun", "pargraph", "QuantLib", "tqdm", "yfinance"):
        importlib.import_module(module)
    time.sleep(1)
    return {"task": task, "host": socket.gethostname(),
            "scaler": version("opengris-scaler"), "numpy": version("numpy"),
            "parfun": version("opengris-parfun"), "pargraph": version("pargraph")}

with Client(address=scheduler_address, object_storage_address=object_storage_address) as client:
    futures = [client.submit(check_worker, task) for task in range(4)]
    results = [future.result() for future in futures]
for result in results:
    print(result)
assert len(results) == 4
print("Notebook worker check passed")
